In [0]:
# Cell 1 — Install dependencies
%pip install torch torchvision pillow
dbutils.library.restartPython()

In [0]:
# Cell 2 — Load model and label mapping from Volume

import json
import torch
import torchvision.models as models

label_mapping_path = "/Volumes/main/default/training_imgs_vol/models/label_mapping.json"
model_weights_path = "/Volumes/main/default/training_imgs_vol/models/resnet50_claim_model.pt"

with open(label_mapping_path, "r") as f:
    mapping = json.load(f)

label2id = mapping["label2id"]
id2label = {int(k): v for k, v in mapping["id2label"].items()}
num_classes = len(label2id)

device = torch.device("cpu")

model = models.resnet50(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load(model_weights_path, map_location=device))
model.eval()

print(f"Model loaded — classes: {id2label}")

In [0]:
# Cell 3 — Define prediction UDF and load the model from the Volume path directly inside the UDF on each call
import io
import torch
import torchvision.transforms as transforms
from PIL import Image
from pyspark.sql.functions import pandas_udf, col
import pandas as pd

MODEL_PATH = "/Volumes/main/default/training_imgs_vol/models/resnet50_claim_model.pt"
LABEL_MAPPING_PATH = "/Volumes/main/default/training_imgs_vol/models/label_mapping.json"
NUM_CLASSES = num_classes  # already defined in Cell 2

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

@pandas_udf("struct<label:string, confidence:float>")
def predict_damage_udf(content_series: pd.Series) -> pd.DataFrame:
    import torch
    import torchvision.models as models
    from PIL import Image
    import io
    import json

    # Load model from Volume inside UDF — no broadcasting needed
    with open(LABEL_MAPPING_PATH, "r") as f:
        mapping = json.load(f)
    id2label_local = {int(k): v for k, v in mapping["id2label"].items()}

    m = models.resnet50(weights=None)
    m.fc = torch.nn.Linear(m.fc.in_features, NUM_CLASSES)
    m.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device("cpu")))
    m.eval()

    results = []
    for content in content_series:
        try:
            image = Image.open(io.BytesIO(content)).convert("RGB")
            tensor = transform(image).unsqueeze(0)
            with torch.no_grad():
                outputs = m(tensor)
                probs = torch.softmax(outputs, dim=1)
                confidence, pred_idx = torch.max(probs, dim=1)
            label = id2label_local[pred_idx.item()]
            conf = round(confidence.item(), 4)
        except Exception:
            label = "unknown"
            conf = 0.0
        results.append({"label": label, "confidence": conf})

    return pd.DataFrame(results)

print("Prediction UDF defined")

In [0]:
# Cell 4 — Run predictions on claim images


claim_images = spark.read.table("workspace.silver.claim_images")

predicted = (
    claim_images
    .withColumn("damage_prediction", predict_damage_udf(col("content")))
)

display(predicted.select("image_name", "damage_prediction").limit(10))

In [0]:
# Cell 5 — Join predictions with claims and policy data

from pyspark.sql.functions import regexp_extract

# Read all tables
claim_images = spark.read.table("workspace.silver.claim_images")
claims = spark.read.table("workspace.silver.claims")

policies = (
    spark.read.table("workspace.silver.policies")
    .withColumnRenamed("CUST_ID", "customer_id")
    .withColumnRenamed("CHASSIS_NO", "chassis_no_policy")  # rename to avoid ambiguity
    .withColumnRenamed("SUM_INSURED", "sum_insured")
    .withColumnRenamed("POL_EFF_DATE", "pol_eff_date")
    .withColumnRenamed("POL_EXPIRY_DATE", "pol_expiry_date")
)

telematics = spark.read.table("workspace.gold.aggregated_telematics")

# Read image metadata CSV — contains image_name, claim_no, chassis_no
image_metadata = (
    spark.read
    .option("header", "true")
    .csv("/Volumes/main/default/claims_vol/claims/metadata/image_metadata.csv")
)

# Step 1 — join predictions with metadata to get claim_no and chassis_no
predicted_with_metadata = (
    predicted
    .join(image_metadata, "image_name", "left")
)

# Step 2 — join with claims to get severity, policy_no, total etc.
with_claims = predicted_with_metadata.join(claims, "claim_no", "left")

# Step 3 — join with policies using policy_no
with_policies = with_claims.join(policies, "policy_no", "left")

# Step 4 — join telematics on chassis_no from image_metadata (not from policies)
full = with_policies.join(telematics, "chassis_no", "left")

display(full.select(
    "image_name",
    "claim_no",
    "chassis_no",
    "damage_prediction",
    "severity",
    "policy_no",
    "sum_insured",
    "total",
    "pol_eff_date",
    "pol_expiry_date",
    "telematics_speed",
    "telematics_latitude",
    "telematics_longitude"
).limit(10))

In [0]:
# Cell 6 — Create rules table

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.silver.claims_rules (
        rule_id BIGINT GENERATED ALWAYS AS IDENTITY,
        rule STRING,
        check_name STRING,
        check_code STRING,
        check_severity STRING,
        is_active BOOLEAN
    )
""")

def insert_rule(rule, name, code, severity, is_active):
    escaped_code = code.replace("'", "\\'")
    spark.sql(f"""
        INSERT INTO workspace.silver.claims_rules
            (rule, check_name, check_code, check_severity, is_active)
        VALUES ('{rule}', '{name}', '{escaped_code}', '{severity}', {is_active})
    """)

print("Rules table ready")

In [0]:
# Cell 7 — Define rules

# ── Rule Group 1: Policy Coverage ────────────────────────────────

valid_policy_date = """
CASE
    WHEN pol_eff_date <= claim_date AND pol_expiry_date >= claim_date
    THEN 'Policy active at claim date'
    ELSE 'Policy not active at claim date'
END
"""
insert_rule("policy coverage", "valid_date", valid_policy_date, "HIGH", True)

valid_claim_amount = """
CASE
    WHEN sum_insured >= total
    THEN 'Claim within policy coverage'
    ELSE 'Claim exceeds policy coverage'
END
"""
insert_rule("policy coverage", "valid_amount", valid_claim_amount, "HIGH", True)


# ── Rule Group 2: Damage Assessment ──────────────────────────────

severity_match = """
CASE
    WHEN severity = 'Total Loss'    AND damage_prediction.label = 'major' THEN 'Severity matches report'
    WHEN severity = 'Major Damage'  AND damage_prediction.label = 'minor' THEN 'Severity matches report'
    WHEN severity = 'Minor Damage'  AND damage_prediction.label = 'ok'    THEN 'Severity matches report'
    WHEN severity = 'Trivial Damage'AND damage_prediction.label = 'ok'    THEN 'Severity matches report'
    ELSE 'Severity does not match'
END
"""
insert_rule("damage assessment", "reported_severity_check", severity_match, "HIGH", True)


# ── Rule Group 3: Scene Recreation ───────────────────────────────

speed_check = """
CASE
    WHEN telematics_speed IS NULL         THEN 'No telematics data'
    WHEN telematics_speed <= 0            THEN 'Invalid speed'
    WHEN telematics_speed <= 45           THEN 'Normal speed'
    ELSE                                       'High speed at incident'
END
"""
insert_rule("scene recreation", "speed_check", speed_check, "HIGH", True)

location_check = """
CASE
    WHEN telematics_latitude IS NULL OR telematics_longitude IS NULL
    THEN 'No location data'
    ELSE 'Location data available'
END
"""
insert_rule("scene recreation", "location_check", location_check, "MEDIUM", True)


# ── Final Decision ────────────────────────────────────────────────

fund_release = """
CASE
    WHEN reported_severity_check = 'Severity matches report'
     AND valid_amount            = 'Claim within policy coverage'
     AND valid_date              = 'Policy active at claim date'
     AND speed_check            != 'High speed at incident'
    THEN 'Release funds'
    ELSE 'Claim needs investigation'
END
"""
insert_rule("decision", "fund_release", fund_release, "HIGH", True)

print("All rules inserted")

In [0]:
# Cell 8 — Apply rules and write Gold table

from pyspark.sql.functions import expr

df = full

# Drop duplicate columns that exist across multiple joined tables
cols_to_drop = ["updated_at", "created_at", "chassis_no_policy", "customer_id"]
df = df.drop(*cols_to_drop)

rules = (
    spark.sql("SELECT * FROM workspace.silver.claims_rules WHERE is_active = true ORDER BY rule_id")
    .collect()
)

for rule in rules:
    print(f"Applying: {rule.rule} → {rule.check_name}")
    df = df.withColumn(rule.check_name, expr(rule.check_code))

(df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.claim_insights")
)

print("Gold table written: workspace.gold.claim_insights")

In [0]:
# Cell 9 — Preview results

display(
    spark.sql("""
        SELECT
            claim_no,
            policy_no,
            severity,
            damage_prediction.label   AS predicted_severity,
            damage_prediction.confidence AS model_confidence,
            valid_date,
            valid_amount,
            reported_severity_check,
            speed_check,
            location_check,
            fund_release
        FROM workspace.gold.claim_insights
        ORDER BY fund_release DESC
    """)
)